In [1]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration


In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

# Data Preprocessing

In [7]:
import re
def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    return text
    

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenization

In [9]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")


In [10]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] 
    return inputs

In [11]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [12]:
type(train_dataset)
type(val_dataset)

list

# Working with our model

In [13]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

C:\Users\Soumadipta Konar\anaconda3\envs\torchgpu\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Soumadipta Konar\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [16]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.598055,0.381354
2,0.396975,0.359468
3,0.374069,0.354952
4,0.361717,0.350415
5,0.355435,0.349755
6,0.351790,0.349234


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9063401133219401, metrics={'train_runtime': 764.8288, 'train_samples_per_second': 31.38, 'train_steps_per_second': 3.922, 'total_flos': 3248203235328000.0, 'train_loss': 0.9063401133219401, 'epoch': 6.0})

In [19]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [20]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [21]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) 

    
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) 
    return summary

In [22]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai technology continues to expand rapidly across industries, from healthcare to finance and education. ai adoption has significantly increased over the past few years.
